# Noise aware Greenberger–Horne–Zeilinger (GHZ) state with Dynamical Decoupling, Pauli-checks, and readout error mitigation

In [1]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

## References

Big cats: entanglement in 120 qubits and beyond - https://doi.org/10.48550/arXiv.2510.09520

Achieving Computational Gains with Quantum Error-Correction Primitives: Generation of Long-Range Entanglement Enhanced by Error Detection - https://doi.org/10.1103/PRXQuantum.6.020331

## Frontmatter

In [ ]:
import time
import numpy as np
from qiskit import *
from qiskit.circuit import ClassicalRegister, Gate, QuantumCircuit
from qiskit.circuit.library import RZGate, XGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (ALAPScheduleAnalysis, PadDynamicalDecoupling, RemoveBarriers)
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import hellinger_fidelity
from qiskit.visualization.gate_map import plot_gate_map

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler, Batch

# Measurement mitigation
import mthree
from mthree.utils import final_measurement_mapping

import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

In [ ]:
import json
from datetime import datetime

def default(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }
    raise TypeError(f"Cannot serialize {type(obj)}")

def object_hook(obj):
    if obj.get("__type__") == "datetime":
        return datetime.fromisoformat(obj["value"])
    return obj

data = {
    "name": "test",
    "created": datetime.now(),
}

In [ ]:

def active_qubits(circ):
    """Compute which physical qubits are actively used in the circuit

    Parameters:
        circ (QuantumCircuit): Input circuit

    Returns:
        list: Active qubits
    """
    active_qubits = set()
    for inst in circ.data:
        if inst.operation.name != "delay" and inst.operation.name != "barrier":
            for qubit in inst.qubits:
                q = circ.find_bit(qubit).index
                active_qubits.add(q)
    return list(active_qubits)


def active_gates(circ):
    """Compute which 2Q gates (edges) are actively used in the circuit

    Parameters:
        circ (QuantumCircuit): Input circuit

    Returns:
        set: Tuples of qubit indices for gates
    """
    used_2q_gates = set()
    for inst in circ:
        if inst.operation.num_qubits == 2:
            qs = inst.qubits
            qs = sorted([circ.find_bit(q).index for q in qs])
            used_2q_gates.add(tuple(sorted(qs)))
    return used_2q_gates


def get_bad_edges(target, two_q_gate_name, threshold):
    """Compute set of 'bad edges' defined by supplied gate error threshold

    Parameters:
        target (Target): Target for backend
        two_q_gate_name (str): Name of the 2Q gate being used
        threshold (float): Error value above which a gate is "bad"

    Returns:
        list: List of bad device edges
    """
    undirected_edges = []
    for edge in target.build_coupling_map().get_edges():
        if (edge[1], edge[0]) not in undirected_edges:
            undirected_edges.append(edge)
    edges = undirected_edges
    two_q_errors = {}
    for edge in edges:
        try:
            error = target[two_q_gate_name][edge].error
        except KeyError:
            error = target[two_q_gate_name][(edge[1], edge[0])].error
        two_q_errors[edge] = error
    worst_edges = sorted(two_q_errors.items(), key=lambda x: x[1], reverse=True)
    return [list(edge) for edge, error in worst_edges if error > threshold]


def get_bad_nodes(target, threshold):
    """Compute set of 'bad nodes' (qubits) defined by supplied measurement error threshold
    
    Parameters:
        target (Target): Target for backend
        threshold (float): Error value above which a measurement is "bad"

    Returns:
        list: List of bad device nodes
    """
    meas_errors = {}
    for node in range(target.num_qubits):
        meas_errors[node] = target["measure"][(node,)].error
    worst_nodes = sorted(meas_errors.items(), key=lambda x: x[1], reverse=True)
    return [node for node, error in worst_nodes if error > threshold]


def circuit_errors(target, circ, error_type, two_q_gate_name=None):
    """Circuit errors helper function
    """
    two_q_errors = []
    meas_errors = []

    for edge in active_gates(circ):
        try:
            error = target[two_q_gate_name][edge].error
        except KeyError:
            error = target[two_q_gate_name][(edge[1], edge[0])].error
        two_q_errors.append(error)

    for qubits in active_qubits(circ):
        meas_errors.append(target["measure"][(qubits,)].error)
    if error_type == "two-q":
        return two_q_errors
    else:
        return meas_errors


def parallel_ghz(root, num_qubits, backend, bad_edges=[], skip=[]):
    """Construct a GHZ state with maximum parallism provided
    by the device topology using a breadth-first search

    Parameters:
        root (int): Index of the starting qubit
        num_qubits (int): Number of qubits in the GHZ state
        backend (IBMBackend): Target device
        bad_edges (list): List of bad edges in the topology
        skip (list): List of physical qubits to skip when building circuit

    Returns:
        QuantumCircuit: Circuit encoding GHZ state
    """
    import copy
    from collections import defaultdict, deque
    cmap = backend.configuration().coupling_map
    untouched_edges = copy.deepcopy(cmap)
    untouched_edges = [
        edge
        for edge in untouched_edges
        if not (
            edge in bad_edges
            or [edge[1], edge[0]] in bad_edges
            or edge[0] in skip
            or edge[1] in skip
        )
    ]
    adj_list = defaultdict(list)  # Convert edge list to an adjacency list
    for u, v in untouched_edges:
        adj_list[u].append(v)
        adj_list[v].append(u)  # Since it's an undirected graph, add both directions
    qc = QuantumCircuit(backend.configuration().num_qubits)
    qc.h(root)
    visited = set()
    queue = deque([root])
    visited.add(root)
    visited_nodes = [root]  # Start with the root node
    while queue and len(visited_nodes) < num_qubits:
        node = queue.popleft()
        # Explore neighbors
        for neighbor in adj_list[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                qc.cx(node, neighbor)
                queue.append(neighbor)
                visited_nodes.append(neighbor)
                if len(visited_nodes) == num_qubits:
                    break
    return qc


def checked_ghz_circuit(backend, num_ghz_qubits, thresh_two_q=0.01, thresh_meas=0.08):
    """Noise aware GHZ circuit with Pauli-checks and Dynamical Decoupling inserted

    Parameters:
        backend (IBMBackend): Target device
        num_ghz_qubits (int): Number of qubits in GHZ state
        thresh_two_q (float): 2Q gate error threshold
        thresh_meas (float): Measurement error threshold

    Returns:
        QuantumCircuit: GHZ circuit with checks

    Internal variable descriptions:
        ROOT: root for BFS search
        SKIP: nodes to intentionally skip so that
            we have a better chance for finding checks
        thresh_two_q: exclude from BFS those edges whose
            CZ error is worse than this threshold
        thresh_meas: exclude from BFS those ndoes whose
            measurement error is worse than this threshold
        MAX_SKIPS: # at most how many qubits to skip
            (i.e. block) while making GHZ (in addition
            to the bad ones and the ones forced to skip above)
        SHUFFLES: how many times to try removing nodes for checks
    """
    ROOT = None
    SKIP = []
    MAX_SKIPS = 7
    SHUFFLES = 3000
    MAX_DEPTH_INCREASE = 3

    coupling_map = backend.target.build_coupling_map()
    two_q_gate_name = {"ecr", "cz", "cx"}.intersection(set(backend.basis_gates)).pop()

    # Find bad edges of topology
    bad_edges = get_bad_edges(
        backend.target, threshold=thresh_two_q, two_q_gate_name=two_q_gate_name
    )
    # Find bad nodes (qubits) based on measurement error
    bad_nodes = get_bad_nodes(backend.target, threshold=thresh_meas)

    # search for the best root (yieling the shallowest GHZ)
    if ROOT is None:
        best_root = -1
        base_depth = 100
        for root in range(backend.num_qubits):
            qc = parallel_ghz(root, num_ghz_qubits, backend, bad_edges, SKIP)
            ghz_qubits = active_qubits(qc)
            if len(ghz_qubits) != num_ghz_qubits:
                continue
            depth = qc.depth(lambda x: x.operation.num_qubits == 2)
            if depth < base_depth:
                best_root = root
                base_depth = depth
        ROOT = best_root

    # build a ghz starting at the best root
    qc = parallel_ghz(ROOT, num_ghz_qubits, backend, bad_edges, SKIP + bad_nodes)
    ghz_qubits = active_qubits(qc)
    base_depth = qc.depth(lambda x: x.operation.num_qubits == 2)

    if len(ghz_qubits) != num_ghz_qubits:
        print(f"Thresholds (two_q, meas): ({thresh_two_q}, {thresh_meas})")
        print(
            f"Qubits in constructed circuit ({len(ghz_qubits)}) !="
            f" qubits requested ({num_ghz_qubits})."
        )
        return None

    # remove random nodes from the ghz and build from the root again in order to increase checks
    degree_two_nodes = [
        i
        for i in ghz_qubits
        if all(n in ghz_qubits for n in coupling_map.neighbors(i))
        and len(coupling_map.neighbors(i)) >= 2
    ]

    # Randomly search for better GHZ and check layouts based on number of checks
    best_covered_fraction = 0
    best_qc = qc
    best_checks = []
    for num_skips in range(MAX_SKIPS):
        for _ in range(SHUFFLES):
            skip = SKIP + list(np.random.choice(degree_two_nodes, num_skips))
            qc = parallel_ghz(
                ROOT, num_ghz_qubits, backend, bad_edges, skip + bad_nodes
            )
            depth = qc.depth(lambda x: x.operation.num_qubits == 2)
            ghz_qubits = active_qubits(qc)
            if len(ghz_qubits) != num_ghz_qubits:
                continue
            checks = []
            for i in range(backend.num_qubits):
                neighbors = [n for n in coupling_map.neighbors(i) if n in ghz_qubits]
                if (i not in ghz_qubits and len(neighbors) >= 2
                    and not any(
                        [
                            [neighbor, i] in bad_edges or [i, neighbor] in bad_edges
                            for neighbor in neighbors
                        ]
                    )
                ):
                    checks.append(i)
                    qc.cx(neighbors[0], i)
                    qc.cx(neighbors[1], i)

                if len(checks) > best_covered_fraction:
                    best_covered_fraction = len(checks)
                    best_qc = qc
                    best_checks = checks

    qc = best_qc
    checks = best_checks
    ghz_qubits = [q for q in active_qubits(qc) if q not in checks]
    if len(ghz_qubits) != num_ghz_qubits:
        raise Exception("No GHZ found. Relax error thresholds.")

    # add measurements of ghz qubits and check qubits into their respective registers
    c1 = ClassicalRegister(len(ghz_qubits), "c1")
    qc.add_register(c1)
    for q, c in zip(ghz_qubits, c1):
        qc.measure(q, c)
    if len(checks) > 0:
        c2 = ClassicalRegister(len(checks), "c2")
        qc.add_register(c2)
        for q, c in zip(checks, c2):
            qc.measure(q, c)

    # transpile the circuit for the backend
    pm = generate_preset_pass_manager(
        target=backend.target,
        layout_method="trivial",
        optimization_level=2,
        routing_method="none",
    )
    # Add XY4 dynamical decoupling to qubit idle periods
    pm.scheduling = PassManager(
        [
            RemoveBarriers(),
            ALAPScheduleAnalysis(target=backend.target),
            PadDynamicalDecoupling(
                dd_sequence=[XGate(), RZGate(-np.pi), XGate(), RZGate(np.pi)],
                spacing=[1 / 4, 1 / 2, 0, 0, 1 / 4],
                target=backend.target,
            ),
        ]
    )

    circuit = pm.run(qc)
    circuit.metadata = {
        "ghz_qubits": ghz_qubits,
        "checks": checks,
        "depth": depth,
        "root": ROOT,
        "skip": SKIP,
        "thresh_cz": thresh_two_q,
        "thresh_meas": thresh_meas,
    }

    return circuit

## Load service and select device

In [ ]:
service = QiskitRuntimeService()

In [ ]:
backend = service.backend("ibm_chicago")

## Generate circuit

In [ ]:
N = 80
st = time.perf_counter()
qc = checked_ghz_circuit(backend, N)
ft = time.perf_counter()
circuit_build_time = ft - st
print(circuit_build_time)

In [ ]:
root = qc.metadata['root']
ghz_qubits = qc.metadata['ghz_qubits']
checks = qc.metadata["checks"]

In [ ]:
qubit_colors = ['#6929c4' if kk in ghz_qubits else '#9f1853' if kk in checks else "#d3d3d3" for kk in range(backend.num_qubits)]

line_colors = []
for edge in backend.configuration().coupling_map:
    if edge[0] in ghz_qubits and edge[1] in ghz_qubits:
        line_colors.append('#6929c4')
    elif (edge[0] in ghz_qubits and edge[1] in checks) or (edge[1] in ghz_qubits and edge[0] in checks):
        line_colors.append('#9f1853')
    else:
        line_colors.append('#d3d3d3')

plot_gate_map(backend, qubit_color=qubit_colors, line_color=line_colors)

## Calibrate readout mitigation

In [ ]:
mit = mthree.M3Mitigation(backend)

In [ ]:
mit_jobs = mit.cals_from_system()

## Execute circuit on device

In [ ]:
sampler = Sampler(backend)

In [ ]:
job = sampler.run([qc], shots=10_000)

## Post-process results

In [ ]:
ans = {"0"*N: 0.5, "1"*N: 0.5}

In [ ]:
ghz_bits = job.result()[0].data.c1
check_bits = job.result()[0].data.c2

In [ ]:
raw_counts = ghz_bits.get_counts()
ghz_bitstrings = ghz_bits.get_bitstrings()
check_bitstrings = check_bits.get_bitstrings()

In [ ]:
checked_counts = {}
for idx, check_str in enumerate(check_bitstrings):
    if not int(check_str):
        if ghz_bitstrings[idx] in checked_counts:
            checked_counts[ghz_bitstrings[idx]] += 1
        else:
            checked_counts[ghz_bitstrings[idx]] = 1

In [ ]:
st = time.perf_counter()
mit_counts = mit.apply_correction(checked_counts, ghz_qubits)
ft = time.perf_counter()
m3_time = ft - st
print(m3_time)

In [ ]:
raw_fid = hellinger_fidelity(raw_counts, ans)
raw_fid

In [ ]:
hellinger_fidelity(checked_counts, ans)

In [ ]:
mit_fid = hellinger_fidelity(mit_counts.nearest_probability_distribution(), ans)
mit_fid

In [ ]:
results = {'name': 'ghz', 'backend': backend.name, 'datetime': str(job.creation_date),
           'backend_properties': json.dumps(backend.properties().to_dict(), default=default),
           'shots': 10_000, 'runtime': job.usage(), 'mit_runtime': sum(jj.usage() for jj in mit_jobs),
           'qubits': N,
           'm3_single_qubit_cals': [mm.tolist() for mm in mit.single_qubit_cals],
           'm3_time': m3_time,
           'fidelity':hellinger_fidelity(raw_counts, ans), 
           'mitigated_fidelity':mit_fid}

In [ ]:
with open("ghz_results.json", "w") as f:
        json.dump(results, f)